# System Design Fundamentals — Hands-On

**Software Engineering · Week 02**

Offline simulations of request flow, ports/adapters, queues, caching, rate limiting, and reliability primitives.

## 0. Client-server request as local function calls

In [ ]:
from dataclasses import dataclass
from collections import OrderedDict, deque, defaultdict

@dataclass(frozen=True)
class Request:
    client_id: str
    path: str
    payload: dict

def server_handle(req):
    return {"status": 200, "body": f"handled {req.path} for {req.client_id}"}

print(server_handle(Request("client-a", "/tickets", {"q": "latency"})))

## 1. Hexagonal core with an output port

In [ ]:
class TicketRepository:
    def save(self, row):
        raise NotImplementedError

class InMemoryRepo(TicketRepository):
    def __init__(self):
        self.rows = []
    def save(self, row):
        self.rows.append(row)
        return len(self.rows)

class CreateTicketUseCase:
    def __init__(self, repo: TicketRepository):
        self.repo = repo
    def execute(self, customer, text):
        if not text.strip():
            raise ValueError("text required")
        return self.repo.save({"customer": customer, "text": text})

repo = InMemoryRepo()
print(CreateTicketUseCase(repo).execute("acme", "cache miss storm"), repo.rows)

## 2. Event bus and queue fanout

In [ ]:
queue = deque()
subscribers = defaultdict(list)

def subscribe(event_type, handler):
    subscribers[event_type].append(handler)

def publish(event_type, payload):
    queue.append((event_type, payload))

def drain():
    while queue:
        event_type, payload = queue.popleft()
        for handler in subscribers[event_type]:
            handler(payload)

seen = []
subscribe("TicketCreated", lambda e: seen.append(("index", e["id"])))
subscribe("TicketCreated", lambda e: seen.append(("notify", e["customer"])))
publish("TicketCreated", {"id": 1, "customer": "acme"})
drain()
print(seen)

## 3. Idempotent consumer for at-least-once delivery

In [ ]:
processed = set()
side_effects = []

def handle_payment(event):
    key = event["idempotency_key"]
    if key in processed:
        return "duplicate ignored"
    processed.add(key)
    side_effects.append(f"ship:{event['order_id']}")
    return "processed"

event = {"idempotency_key": "evt-1", "order_id": "ord-7"}
print(handle_payment(event), handle_payment(event), side_effects)

## 4. LRU cache-aside

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()
    def get(self, key, loader):
        if key in self.data:
            self.data.move_to_end(key)
            return self.data[key], "hit"
        value = loader(key)
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)
        return value, "miss"

cache = LRUCache(2)
for key in ["u1", "u2", "u1", "u3"]:
    print(key, cache.get(key, lambda k: {"profile": k}), list(cache.data))

## 5. Token-bucket rate limit with simulated time

In [ ]:
@dataclass
class TokenBucket:
    capacity: float
    refill_per_second: float
    tokens: float = 0.0
    updated_at: float = 0.0
    def __post_init__(self):
        self.tokens = self.capacity
    def allow(self, now, cost=1):
        self.tokens = min(self.capacity, self.tokens + max(0, now - self.updated_at) * self.refill_per_second)
        self.updated_at = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False

bucket = TokenBucket(2, 1)
print([bucket.allow(t) for t in [0, 0, 0, 1.0, 1.1, 2.0]])

## 6. Circuit breaker stops repeated failing calls

In [ ]:
class CircuitBreaker:
    def __init__(self, failure_threshold):
        self.failure_threshold = failure_threshold
        self.failures = 0
        self.open = False
    def call(self, fn):
        if self.open:
            return "skipped: circuit open"
        try:
            result = fn()
            self.failures = 0
            return result
        except Exception as exc:
            self.failures += 1
            if self.failures >= self.failure_threshold:
                self.open = True
            return f"failed: {type(exc).__name__}"

breaker = CircuitBreaker(2)
bad = lambda: (_ for _ in ()).throw(TimeoutError("dependency slow"))
print(breaker.call(bad), breaker.call(bad), breaker.call(lambda: "ok"))

## 7. Capacity math: queue lag under burst

In [ ]:
arrival_per_sec = [10, 30, 5, 5]
worker_capacity = 12
lag = 0
for sec, arrivals in enumerate(arrival_per_sec, 1):
    lag = max(0, lag + arrivals - worker_capacity)
    print(f"sec={sec} arrivals={arrivals} lag={lag}")
print("lag is the backpressure signal")

## Exercises
1. Add a dead-letter queue after three handler failures.
2. Extend the cache with TTL and explicit invalidation.
3. Compute retry-after for the token bucket when a request is rejected.
4. Add a half-open state to the circuit breaker.

## Links
- Literature note: `02 Literature Notes/Software Engineering/System Design Fundamentals`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 02 Token Bucket Rate Limiter`, `.../SE Week 02 In-Memory LRU Cache`
- MOC: `06 Maps of Content/Software Engineering Concepts`